In [ ]:
'''
Santa 2026 · Elf Calculator

V1 · Verified baseline and local optimization

This notebook shortens existing programs and measures the instruction savings
for each puzzle. The pinned public baseline is Ishan Karnick's Frontier Programs
at 128,486 instructions. The optimizer is implemented in this notebook.

Public baseline
https://www.kaggle.com/code/ishankarnick/santa-2026-frontier-programs-128-486

Data and calculator rules
https://www.kaggle.com/competitions/santa-2026/data

Scoring and submission format
https://www.kaggle.com/competitions/santa-2026/overview/evaluation
'''

## 1 · Configuration

In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import tempfile
from dataclasses import dataclass
from pathlib import Path
from time import perf_counter


'''
For a local run, put puzzles.csv and sample_submission.csv in santa-2026/data.
To use the 128,486 baseline, save the Frontier Programs notebook's submission.csv
as data/submission_frontier_128486.csv. On Kaggle, attach the notebook output
or set SANTA_FRONTIER_CSV to its exact path.
'''

@dataclass(frozen=True)
class Config:
    data_dir: Path
    frontier_csv: Path | None
    output_csv: Path
    rewrite_rounds: int = 2

    @classmethod
    def discover(cls) -> Config:
        root = Path.cwd()
        repo_path = root / 'competitions' / 'public-comp' / 'santa-2026'
        if repo_path.is_dir():
            root = repo_path

        kaggle_input = Path('/kaggle/input')
        data_dir = root / 'data'
        output_dir = root / 'working'
        frontier = data_dir / 'submission_frontier_128486.csv'

        if kaggle_input.is_dir():
            matches = sorted(kaggle_input.rglob('puzzles.csv'))
            if len(matches) != 1:
                raise FileNotFoundError('Attach exactly one Santa 2026 competition input')
            data_dir = matches[0].parent
            output_dir = Path('/kaggle/working')
            frontier = next(
                (path for path in sorted(kaggle_input.rglob('submission.csv'))
                 if 'santa-2026-frontier-programs-128-486' in path.parts),
                None,
            )

        if path := os.environ.get('SANTA_FRONTIER_CSV'):
            frontier = Path(path)
        elif frontier is not None and not frontier.is_file():
            frontier = None

        return cls(data_dir, frontier, output_dir / 'submission.csv')


FRONTIER_SHA256 = '67b9917d4102136152a48fbe8b5a3dcae5632e6784ab5e4ae0424d84db4761f9'
FRONTIER_SCORE = 128_486
MAX_INSTRUCTIONS = 4_096
MAX_CELL_BYTES = 131_072
SUBMISSION_COLUMNS = ['id', 'program']

Instruction = tuple[str, int, int, int]
Program = tuple[Instruction, ...]
Registers = tuple[int | None, ...]

## 2 · Puzzle and exact replay

In [ ]:
@dataclass(frozen=True)
class Puzzle:
    id: str
    starting_values: tuple[int, ...]
    targets: frozenset[int]
    value_limit: int
    registers: int

    @classmethod
    def from_row(cls, row: dict[str, str]) -> Puzzle:
        return cls(
            row['id'],
            tuple(json.loads(row['starting_values'])),
            frozenset(json.loads(row['targets'])),
            int(row['value_limit']),
            int(row['registers']),
        )


@dataclass
class Trace:
    before: list[Registers]
    results: list[int]
    dependencies: list[tuple[int, ...]]
    last_use: list[int]
    first_targets: dict[int, int]


class Calculator:
    '''Replay with Python integers to preserve exact division and large products.'''

    def replay(self, puzzle: Puzzle, program: Program, *, trace: bool = False) -> Trace | None:
        if len(program) > MAX_INSTRUCTIONS:
            raise ValueError(f'{puzzle.id}: too many instructions')

        registers = list(puzzle.starting_values) + [None] * (
            puzzle.registers - len(puzzle.starting_values)
        )
        missing = set(puzzle.targets).difference(puzzle.starting_values)
        history = Trace([], [], [], [-1] * len(program), {}) if trace else None
        producers = [-1] * puzzle.registers

        for index, instruction in enumerate(program):
            if len(instruction) != 4:
                raise ValueError(f'{puzzle.id}: malformed instruction {index + 1}')
            op, left, right, dest = instruction
            if any(type(reg) is not int or not 0 <= reg < puzzle.registers
                   for reg in (left, right, dest)):
                raise ValueError(f'{puzzle.id}: invalid register at step {index + 1}')

            a, b = registers[left], registers[right]
            if a is None or b is None:
                raise ValueError(f'{puzzle.id}: undefined source at step {index + 1}')
            if op == '+':
                value = a + b
            elif op == '-':
                value = a - b
            elif op == '*':
                value = a * b
            elif op == '/' and a % b == 0:
                value = a // b
            else:
                raise ValueError(f'{puzzle.id}: illegal operation at step {index + 1}')
            if not 1 <= value <= puzzle.value_limit:
                raise ValueError(f'{puzzle.id}: value outside bounds at step {index + 1}')

            if history is not None:
                history.before.append(tuple(registers))
                history.results.append(value)
                deps = tuple(p for p in (producers[left], producers[right]) if p >= 0)
                history.dependencies.append(deps)
                for parent in deps:
                    history.last_use[parent] = index
                if value in missing:
                    history.first_targets[value] = index
                producers[dest] = index

            registers[dest] = value
            missing.discard(value)

        if missing:
            raise ValueError(f'{puzzle.id}: {len(missing)} uncovered targets')
        return history

## 3 · Local program optimization

In [ ]:
class RecipeFinder:
    '''Invert an operation with known result and one known positive operand.'''

    @staticmethod
    def operands(result: int, right: int, limit: int):
        if result > right:
            yield '+', result - right, True
        if result + right <= limit:
            yield '-', result + right, True
        if right > result:
            yield '-', right - result, False
        if result % right == 0:
            yield '*', result // right, True
        if result * right <= limit:
            yield '/', result * right, True
        if right % result == 0:
            yield '/', right // result, False

    def direct(
        self, result: int, dest: int, by_value: dict[int, int], limit: int,
    ) -> Instruction | None:
        for right, right_reg in by_value.items():
            for op, left, normal_order in self.operands(result, right, limit):
                left_reg = by_value.get(left)
                if left_reg is not None:
                    if normal_order:
                        return op, left_reg, right_reg, dest
                    return op, right_reg, left_reg, dest
        return None

    def find(self, result: int, dest: int, helper: int, before: Registers,
             limit: int, budget: int) -> Program | None:
        available = {value: reg for reg, value in enumerate(before) if value is not None}
        instruction = self.direct(result, dest, available, limit)
        if instruction is not None:
            return (instruction,)
        if budget == 1:
            return None

        '''
        The first write may clobber a source needed by the second operation.
        Enumerate partners from the registers that survive that write.
        '''
        for reg, value in enumerate(before):
            if reg == helper or value is None:
                continue
            for op, intermediate, normal_order in self.operands(result, value, limit):
                first = self.direct(intermediate, helper, available, limit)
                if first is None:
                    continue
                left, right = (helper, reg) if normal_order else (reg, helper)
                return first, (op, left, right, dest)

        square_root = math.isqrt(result)
        for op, intermediate, exact in (
            ('+', result // 2, result % 2 == 0),
            ('*', square_root, square_root ** 2 == result),
        ):
            if exact:
                first = self.direct(intermediate, helper, available, limit)
                if first is not None:
                    return first, (op, helper, helper, dest)
        return None


class ProgramOptimizer:
    def __init__(self, calculator: Calculator, rewrite_rounds: int):
        self.calculator = calculator
        self.rewrite_rounds = rewrite_rounds
        self.recipes = RecipeFinder()

    def prune(self, puzzle: Puzzle, program: Program) -> Program:
        trace = self.calculator.replay(puzzle, program, trace=True)
        required = set(trace.first_targets.values())
        stack = list(required)
        while stack:
            for dep in trace.dependencies[stack.pop()]:
                if dep not in required:
                    required.add(dep)
                    stack.append(dep)
        return tuple(step for index, step in enumerate(program) if index in required)

    def rebuild_pair(
        self, before: Registers, outputs: tuple[tuple[int, int], tuple[int, int]],
        limit: int,
    ) -> Program | None:
        first, second = outputs
        orders = (outputs,) if first[1] == second[1] else (outputs, (second, first))
        for order in orders:
            registers = list(before)
            steps = []
            for value, dest in order:
                available = {v: reg for reg, v in enumerate(registers) if v is not None}
                step = self.recipes.direct(value, dest, available, limit)
                if step is None:
                    break
                steps.append(step)
                registers[dest] = value
            if len(steps) == 2:
                return tuple(steps)
        return None

    def rewrite(self, puzzle: Puzzle, program: Program) -> Program:
        trace = self.calculator.replay(puzzle, program, trace=True)
        target_writes = set(trace.first_targets.values())

        '''
        A replacement must preserve each new target and each write that feeds
        the suffix. Keeping a live write in its original register preserves its readers.
        '''
        for width in (3, 2):
            for end in range(width - 1, len(program)):
                start = end - width + 1
                outputs = tuple(
                    (trace.results[index], program[index][3])
                    for index in range(start, end + 1)
                    if index in target_writes or trace.last_use[index] > end
                )
                if len(outputs) == 1:
                    result, dest = outputs[0]
                    replacement = self.recipes.find(
                        result, dest, program[start][3], trace.before[start],
                        puzzle.value_limit, width - 1,
                    )
                elif width == 3 and len(outputs) == 2:
                    replacement = self.rebuild_pair(
                        trace.before[start], outputs, puzzle.value_limit,
                    )
                else:
                    continue
                if replacement is not None:
                    return program[:start] + replacement + program[end + 1:]
        return program

    def optimize(self, puzzle: Puzzle, program: Program) -> Program:
        current = self.prune(puzzle, program)
        for _ in range(self.rewrite_rounds):
            candidate = self.rewrite(puzzle, current)
            if len(candidate) == len(current):
                break
            current = self.prune(puzzle, candidate)
        self.calculator.replay(puzzle, current)
        return current

## 4 · Submission and per-puzzle report

In [ ]:
class SantaPipeline:
    def __init__(self, config: Config):
        self.config = config
        self.calculator = Calculator()
        self.optimizer = ProgramOptimizer(self.calculator, config.rewrite_rounds)

    def load_puzzles(self) -> dict[str, Puzzle]:
        with (self.config.data_dir / 'puzzles.csv').open(newline='', encoding='utf-8') as file:
            puzzles = [Puzzle.from_row(row) for row in csv.DictReader(file)]
        by_id = {puzzle.id: puzzle for puzzle in puzzles}
        expected_ids = {f'scored_{index:04}' for index in range(1, 1_001)}
        if len(puzzles) != 1_000 or by_id.keys() != expected_ids:
            raise ValueError('Expected each scored puzzle exactly once')
        return by_id

    def read_submission(self, path: Path, puzzles: dict[str, Puzzle]) -> dict[str, Program]:
        programs = {}
        with path.open(newline='', encoding='utf-8') as file:
            reader = csv.DictReader(file)
            if reader.fieldnames != SUBMISSION_COLUMNS:
                raise ValueError(f'{path}: expected id,program columns')
            for row in reader:
                puzzle_id, cell = row['id'], row['program']
                if None in row or cell is None or puzzle_id not in puzzles or puzzle_id in programs:
                    raise ValueError(f'{path}: malformed or duplicate row for {puzzle_id}')
                if len(cell.encode('utf-8')) > MAX_CELL_BYTES:
                    raise ValueError(f'{puzzle_id}: program cell exceeds the byte limit')
                payload = json.loads(cell)
                if type(payload) is not list or any(type(step) is not list for step in payload):
                    raise ValueError(f'{puzzle_id}: expected a JSON array of instructions')
                program = tuple(tuple(step) for step in payload)
                self.calculator.replay(puzzles[puzzle_id], program)
                programs[puzzle_id] = program
        if programs.keys() != puzzles.keys():
            raise ValueError(f'{path}: missing puzzle IDs')
        return programs

    def write_submission(self, programs: dict[str, Program],
                         puzzles: dict[str, Puzzle]) -> None:
        output = self.config.output_csv
        output.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.NamedTemporaryFile(
            mode='w', encoding='utf-8', newline='', dir=output.parent, delete=False,
        ) as file:
            temporary = Path(file.name)
            try:
                writer = csv.writer(file)
                writer.writerow(SUBMISSION_COLUMNS)
                for puzzle_id, program in programs.items():
                    writer.writerow((puzzle_id, json.dumps(program, separators=(',', ':'))))
                file.close()
                if self.read_submission(temporary, puzzles) != programs:
                    raise RuntimeError('Program changed during serialization')
                temporary.replace(output)
            finally:
                temporary.unlink(missing_ok=True)

    def run(self) -> list[tuple[str, int, int]]:
        start = perf_counter()
        puzzles = self.load_puzzles()
        baseline_path = self.config.frontier_csv or self.config.data_dir / 'sample_submission.csv'
        if self.config.frontier_csv is not None:
            digest = hashlib.sha256(baseline_path.read_bytes()).hexdigest()
            if digest != FRONTIER_SHA256:
                raise ValueError(f'{baseline_path}: expected the pinned 128,486 CSV, got {digest}')
        baseline = self.read_submission(baseline_path, puzzles)
        baseline_score = sum(map(len, baseline.values()))
        if self.config.frontier_csv is not None and baseline_score != FRONTIER_SCORE:
            raise ValueError(f'Unexpected public baseline score: {baseline_score}')

        programs = {}
        changes = []
        for puzzle_id, program in baseline.items():
            candidate = self.optimizer.optimize(puzzles[puzzle_id], program)
            programs[puzzle_id] = candidate
            if len(candidate) < len(program):
                changes.append((puzzle_id, len(program), len(candidate)))

        self.write_submission(programs, puzzles)
        score = sum(map(len, programs.values()))
        changes.sort(key=lambda row: row[2] - row[1])
        print(f'Baseline: {baseline_path.name} · {baseline_score:,}')
        print(f'Local score: {score:,} · saved {baseline_score - score:,} instructions')
        print(f'Validated: {len(programs):,} · improved: {len(changes):,}')
        print(f'Elapsed: {perf_counter() - start:.1f}s · {self.config.output_csv}')
        for puzzle_id, before, after in changes[:10]:
            print(f'{puzzle_id}: {before} -> {after}  (-{before - after})')
        return changes

## 5 · Self-check and run

In [ ]:
calculator = Calculator()
optimizer = ProgramOptimizer(calculator, rewrite_rounds=2)

example = Puzzle('example', (3,), frozenset({1, 2, 5, 8}), 9, 48)
official_example = (('/', 0, 0, 1), ('-', 0, 1, 1), ('+', 0, 1, 2), ('+', 0, 2, 3))
calculator.replay(example, official_example)
assert optimizer.optimize(example, official_example + (('/', 0, 0, 4),)) == official_example

two_step = Puzzle('two_step', (2, 3), frozenset({11}), 12, 48)
three_steps = (('+', 0, 1, 2), ('+', 2, 1, 2), ('+', 2, 1, 2))
assert len(optimizer.optimize(two_step, three_steps)) == 2

pair = Puzzle('pair', (2, 3, 4), frozenset({8, 9}), 12, 48)
with_shared_helper = (('+', 0, 1, 3), ('+', 3, 1, 4), ('+', 3, 2, 5))
assert len(optimizer.optimize(pair, with_shared_helper)) == 2

try:
    calculator.replay(example, (('/', 0, 0, 1), ('/', 1, 0, 2)))
except ValueError as error:
    assert 'illegal operation' in str(error)
else:
    raise AssertionError('Non-exact division passed replay')

In [ ]:
config = Config.discover()
changes = SantaPipeline(config).run()